# AuxiliaryASR PER Evaluation

This notebook loads a trained AuxiliaryASR model, prepares the validation dataset, and computes the phoneme error rate (PER) using greedy CTC decoding.

In [ ]:
# check available CUDA devices
import torch
devices = []
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        device_name = torch.cuda.get_device_name(i)
        devices.append({
            'type': 'CUDA',
            'available': True,
            'name': device_name,
            'index': i
        })
else:
    devices.append({'type': 'CUDA', 'available': False, 'name': 'N/A'})
devices

In [ ]:
# change folder into the root of the ASR project
import os

if not os.path.isdir('Configs'):
    %cd ../

!pwd

## Imports and helper utilities

In [ ]:
# import packages, define helper utilities
import os
import yaml
import torch
import pandas as pd
from collections import Counter

from models import ASRCNN
from meldataset import build_dataloader
from token_map import build_token_map_from_data
from text_utils import TextCleaner

def cfg_get_nested(cfg: dict, path, default=None, sep='.'):
    """Get a nested value from a dict using a list of keys or a dot-separated string."""
    if isinstance(path, str):
        keys = path.split(sep) if path else []
    else:
        keys = path

    cur = cfg
    for k in keys:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        else:
            return default
    return cur

def load_token_map_from_config(config):
    token_src = config.get('phoneme_maps_path')
    if not token_src:
        return build_token_map_from_data(
            config.get('train_data'),
            config.get('val_data'),
            config.get('ood_data'),
            apply_asr_tokenizer=True,
        )
    if isinstance(token_src, dict):
        return token_src
    csv = pd.read_csv(token_src, header=None).values
    return {word: index for word, index in csv}

def load_asr_model(model_path, config_path, device):
    with open(config_path) as f:
        config = yaml.safe_load(f)

    token_map = load_token_map_from_config(config)

    model_params = cfg_get_nested(config, 'model_params', {
        'input_dim': 80,
        'hidden_dim': 256,
        'n_token': len(token_map),
        'token_embedding_dim': 512,
        'n_layers': 5,
        'location_kernel_size': 31,
    })
    if 'n_token' not in model_params:
        model_params['n_token'] = len(token_map)

    model = ASRCNN(**model_params)
    checkpoint = torch.load(model_path, map_location='cpu', weights_only=False)
    state_dict = checkpoint.get('model', checkpoint)
    try:
        model.load_state_dict(state_dict)
    except RuntimeError:
        sanitized_state = {k.replace('module.', ''): v for k, v in state_dict.items()}
        model.load_state_dict(sanitized_state)

    model.to(device)
    model.eval()
    return model, config, token_map

def build_dev_dataloader(config, device, batch_size=None, num_workers=None):
    val_csv_path = config.get('val_data')
    if val_csv_path is None:
        raise ValueError("Validation CSV path ('val_data') not found in the config.")

    with open(val_csv_path, 'r', encoding='utf-8') as f:
        raw_lines = [line.rstrip('\n') for line in f]

    path_list = []
    for raw in raw_lines:
        if not raw.strip():
            continue
        parts = raw.split('|')
        if len(parts) == 1:
            continue
        path = parts[0]
        if len(parts) == 2:
            text = parts[1]
            speaker = ''
        else:
            text = '|'.join(parts[1:-1])
            speaker = parts[-1]
        path_list.append([path, text, speaker])

    if batch_size is None:
        batch_size = int(cfg_get_nested(config, 'eval_params.batch_size', cfg_get_nested(config, 'batch_size', 4)))
    if num_workers is None:
        num_workers = int(cfg_get_nested(config, 'dataloader_params.val_num_workers', 2))

    dataset_params = {
        'dict_path': cfg_get_nested(config, 'phoneme_maps_path', 'Data/word_index_dict.txt'),
        'sr': cfg_get_nested(config, 'preprocess_params.sr', 24000),
        'spect_params': cfg_get_nested(
            config,
            'preprocess_params.spect_params',
            {'n_fft': 1024, 'win_length': 1024, 'hop_length': 300},
        ),
        'mel_params': cfg_get_nested(
            config, 'preprocess_params.mel_params', {'n_mels': 80}
        ),
    }

    collate_config = {'return_wave': False}
    dataset_params, collate_config = ensure_auxiliary_collate(config, dataset_params, collate_config)
    device_flag = device.type if isinstance(device, torch.device) else str(device)
    loader = build_dataloader(
        path_list=path_list,
        validation=True,
        batch_size=batch_size,
        num_workers=num_workers,
        device=device_flag,
        collate_config=collate_config,
        dataset_config=dataset_params,
    )
    return loader



def ensure_auxiliary_collate(config, dataset_params, collate_config=None):
    if dataset_params is None:
        dataset_params = {}
    if collate_config is None:
        collate_config = {}
    dataset_params = dict(dataset_params)
    collate_config = dict(collate_config)
    aux_objectives = cfg_get_nested(config, "auxiliary_objectives", {}) or {}
    dataset_params["auxiliary_objectives"] = aux_objectives
    collate_config["auxiliary_objectives"] = aux_objectives
    collate_config["include_auxiliary_targets"] = any(
        obj.get("enabled", False) for obj in aux_objectives.values()
    )
    return dataset_params, collate_config

## Load model, configuration, and validation loader

In [ ]:
checkpoint_dir = 'Checkpoint'
config_path = 'Checkpoint/config.yml'

if not os.path.isdir(checkpoint_dir):
    raise FileNotFoundError(f"Checkpoint directory '{checkpoint_dir}' not found.")

ckpt_files = [f for f in os.listdir(checkpoint_dir) if f.startswith('epoch_') and f.endswith('.pth')]
if not ckpt_files:
    raise FileNotFoundError(f"No checkpoint files found in '{checkpoint_dir}'.")

ckpt_files = sorted(ckpt_files, key=lambda x: int(x.split('_')[-1].split('.')[0]))
model_path = os.path.join(checkpoint_dir, ckpt_files[-1])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, config, token_map = load_asr_model(model_path, config_path, device)

print(f'model --> {model_path}')
print(f'config --> {config_path}')
phoneme_source = config.get('phoneme_maps_path', 'built from dataset')
print(f'dictionary --> {phoneme_source}')

dev_loader = build_dev_dataloader(config, device)
print(f'Validation dataset size: {len(dev_loader.dataset)} samples')

vocab = token_map
if ' ' not in vocab:
    raise KeyError("The vocabulary does not contain the blank symbol ' '.")
BLANK_ID = vocab[' ']
ID2PH = {idx: symbol for symbol, idx in vocab.items()}
print(f'Blank token id: {BLANK_ID}')


## Greedy CTC decoding and PER computation

In [ ]:
# decoding utilities and PER evaluation
from typing import List, Sequence

def ctc_greedy_decode(logits: torch.Tensor, lens: torch.Tensor) -> List[List[int]]:
    """Greedy CTC decoding with collapse and blank removal."""
    if logits.dim() != 3:
        raise ValueError(f"Expected logits of shape (B, T, V), got {tuple(logits.shape)}")

    pred_ids = logits.argmax(-1)
    hyps: List[List[int]] = []
    for b in range(pred_ids.size(0)):
        prev = BLANK_ID
        out: List[int] = []
        T = int(lens[b])
        for t in range(T):
            p = int(pred_ids[b, t])
            if p != BLANK_ID and p != prev:
                out.append(p)
            prev = p
        hyps.append(out)
    return hyps

def edit_distance(a: Sequence[int], b: Sequence[int]) -> int:
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1):
        dp[i][0] = i
    for j in range(len(b) + 1):
        dp[0][j] = j
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + (a[i - 1] != b[j - 1]),
            )
    return dp[-1][-1]

@torch.no_grad()
def eval_per(model: torch.nn.Module, dev_loader, device=None, max_examples: int = 5):
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    tot_err, tot_len = 0, 0
    phoneme_freq: Counter = Counter()
    examples = []
    downsample_factor = 2 ** getattr(model, 'n_down', 1)

    for batch in dev_loader:
        texts, text_lens, mels, mel_lens = batch[:4]
        mels = mels.to(device)
        text_lens = text_lens.to(torch.long)
        mel_lens = mel_lens.to(torch.long)

        outputs = model(mels)
        if isinstance(outputs, dict):
            logits = outputs.get('logits_ctc')
            if logits is None:
                raise KeyError("Model output dict does not contain 'logits_ctc'.")
        elif isinstance(outputs, (tuple, list)):
            logits = outputs[0]
        else:
            logits = outputs

        if logits.dim() != 3:
            raise ValueError(f"Unexpected logits shape: {tuple(logits.shape)}")

        logit_lens = torch.clamp(mel_lens // downsample_factor, min=1, max=logits.size(1))
        hyps = ctc_greedy_decode(logits.cpu(), logit_lens.cpu())

        for hyp, tgt, tgt_len in zip(hyps, texts, text_lens):
            effective_len = int(tgt_len)
            tgt_ids = tgt[:effective_len].tolist()
            tgt_ids = [idx for idx in tgt_ids if idx != BLANK_ID]

            phoneme_freq.update(tgt_ids)
            tot_err += edit_distance(hyp, tgt_ids)
            tot_len += len(tgt_ids)

            if len(examples) < max_examples:
                examples.append({
                    'prediction': hyp,
                    'reference': tgt_ids,
                })

    per = tot_err / max(1, tot_len)
    stats = {
        'total_errors': tot_err,
        'total_phonemes': tot_len,
        'phoneme_frequency': phoneme_freq,
        'examples': examples,
    }
    return per, stats


## Run evaluation

In [ ]:
per, per_stats = eval_per(model, dev_loader, device=device, max_examples=5)
print(f'Dev PER (CTC greedy): {per:.3f}')
print(f'Total phonemes evaluated: {per_stats["total_phonemes"]}')
print(f'Total edit distance: {per_stats["total_errors"]}')


## Inspect sample predictions

In [ ]:
def ids_to_symbols(ids):
    return ' '.join(ID2PH.get(i, f"<unk:{i}>") for i in ids)

for idx, example in enumerate(per_stats["examples"], 1):
    print(f'Sample {idx}')
    print('Reference :', ids_to_symbols(example['reference']))
    print('Prediction:', ids_to_symbols(example['prediction']))
    print('-' * 60)
    if idx >= 5:
        break


In [ ]:
print('Most common phonemes in the validation references:')
for phoneme_id, count in per_stats["phoneme_frequency"].most_common(10):
    symbol = ID2PH.get(phoneme_id, phoneme_id)
    print(f'{symbol}: {count}')
